In [2]:
import numpy as np

"""
Let H_circ be a circulant matrix constructed by vector h of size N,

H_circ = F.conj().T @ Lambda @ F, where F is the unitary N-point DFT matrix.
and Lambda = np.diag(F @ h * np.sqrt(N)) or Lambda = np.diag(np.fft.fft(h)) Note: non-unitary FFT

Therefore, Lambda = F @ H_circ @ F.conj().T = np.diag(F @ h * np.sqrt(N))

Application: By having any H_circ, we can use F to diagnoalize it. 
To this point, we can directly estimate orthogonal channel.

"""

def circulant(x):
    A = np.zeros([x.size, x.size], dtype=int)
    for i in range(x.size):
        A[:,i] = x
        x = np.concat([np.array([x[-1]]), x[:-1]])
    return A

def x_padded(x, N):
    y = np.zeros(N)
    y[:x.size] = x
    return y

N = 5
n_tap = 3
F = np.fft.fft(np.eye(N), norm='ortho')
x = np.random.randint(1, 10, n_tap)
h = x_padded(x, N)
H_circ = circulant(h)
print("H_circ:\n\n{}".format(H_circ))

H_circ:

[[9 0 0 3 9]
 [9 9 0 0 3]
 [3 9 9 0 0]
 [0 3 9 9 0]
 [0 0 3 9 9]]


In [3]:
H_circ = np.complex64(H_circ)
Lambda_ = F @ H_circ @ F.conj().T   # This Gamma_ is diagonal but not equal to h
g = np.diagonal(Lambda_)
Lambda = np.diag(g)

print("Lambda:\n\n{}".format(Lambda))

Lambda:

[[21.         +0.j          0.         +0.j
   0.         +0.j          0.         +0.j
   0.         +0.j        ]
 [ 0.         +0.j          9.35410197-10.3228644j
   0.         +0.j          0.         +0.j
   0.         +0.j        ]
 [ 0.         +0.j          0.         +0.j
   2.64589803 -2.43689772j  0.         +0.j
   0.         +0.j        ]
 [ 0.         +0.j          0.         +0.j
   0.         +0.j          2.64589803 +2.43689772j
   0.         +0.j        ]
 [ 0.         +0.j          0.         +0.j
   0.         +0.j          0.         +0.j
   9.35410197+10.3228644j ]]


In [10]:
# Fouier Transform of h
Fh01 = F @ h * np.sqrt(N)
Fh02 = np.fft.fft(h)

h_est = np.fft.ifft(g)

print("g:\n{}\n".format(g))
print("Fh02:\n{}\n".format(Fh02))
print("h_estimate:\n{}\n".format(h_est))
print("Difference:\n{}".format(np.sum(np.abs(g - Fh01))))

g:
[21.         +0.j          9.35410197-10.3228644j
  2.64589803 -2.43689772j  2.64589803 +2.43689772j
  9.35410197+10.3228644j ]

Fh02:
[21.         +0.j          9.35410197-10.3228644j
  2.64589803 -2.43689772j  2.64589803 +2.43689772j
  9.35410197+10.3228644j ]

h_estimate:
[ 9.0000000e+00+0.j  9.0000000e+00+0.j  3.0000000e+00+0.j
 -1.0658141e-15+0.j  0.0000000e+00+0.j]

Difference:
1.0563036869186399e-14


In [9]:
# Estimate h from Lambda
h_est = np.fft.ifft(g)

print("h_est:\n{}\n".format(h_est))
print("Difference:\n{}".format(np.sum(np.abs(h_est - h))))

h_est:
[ 9.0000000e+00+0.j  9.0000000e+00+0.j  3.0000000e+00+0.j
 -1.0658141e-15+0.j  0.0000000e+00+0.j]

Difference:
3.730349362740526e-15
